In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/application_train.csv")

print("Corrupted DAYS_EMPLOYED count:", (df["DAYS_EMPLOYED"] == 365243).sum())

# Replace 365243 with NaN — it means "not employed"
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

# Create a flag column BEFORE filling — captures "was this person unemployed?"
df["DAYS_EMPLOYED_MISSING"] = df["DAYS_EMPLOYED"].isnull().astype(int)

# Now fill NaN with 0 (unemployed = 0 days employed)
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].fillna(0)

print("After fix — max DAYS_EMPLOYED:", df["DAYS_EMPLOYED"].max())
print("Unemployed applicants flagged:", df["DAYS_EMPLOYED_MISSING"].sum())

Corrupted DAYS_EMPLOYED count: 55374
After fix — max DAYS_EMPLOYED: 0.0
Unemployed applicants flagged: 55374


In [2]:
# Any column missing more than 40% of values is unreliable — drop it
threshold = 0.40
missing_pct = df.isnull().sum() / len(df)

cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()

print(f"Dropping {len(cols_to_drop)} columns with >40% missing data:")
print(cols_to_drop)

df = df.drop(columns=cols_to_drop)
print(f"\nShape after dropping: {df.shape}")

Dropping 49 columns with >40% missing data:
['OWN_CAR_AGE', 'EXT_SOURCE_1', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'TOTALAREA_MODE'

In [3]:
# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

# Numeric: fill with median (robust to outliers)
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Categorical: fill with mode (most common value)
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

# Verify
print("Missing values remaining:", df.isnull().sum().sum())
print("Final shape:", df.shape)

Missing values remaining: 0
Final shape: (307511, 74)


In [4]:
df.to_csv("data/application_train_clean.csv", index=False)
print("✅ Saved to data/application_train_clean.csv")

✅ Saved to data/application_train_clean.csv
